#**Document Aware hYbrid Architecture (DAYA)**

**Copyright 2026 Ronak N Patel**

**Licensed under the Apache License, Version 2.0(the "License");
you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0**

**Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implie**

### **Set up the APIs and run all cells**
### **Have Implemented specific functions compatible with free tier Services Change the functions if you were swap it to some other API services**

### **Refer https://github.com/RoneyBABA/DAYA/blob/main/Colab/README.md fo more**

# **Dependencies**

In [ ]:
!pip install --upgrade torchao peft

In [ ]:
!pip install numpy
!pip install python-pptx xmltodict easyocr chromadb groq docling
!pip install "transformers<5.0.0" "accelerate>=0.26.0"
!pip install --upgrade docling docling-core
!pip install pypdf
!apt-get update
!apt-get install -y libreoffice

In [ ]:
import json
import xmltodict
import fitz
import os
import zipfile
from google.colab import files
from IPython.display import display
from PIL import Image as PILImage
import subprocess
import ipywidgets as widgets
import xml.etree.ElementTree as ET
from pptx import Presentation
import io

In [ ]:
print("\n\n")
!libreoffice --version
print("✅ All packages loaded")

global tree
tree_json = []
doc_op = []
print(os.listdir())

In [ ]:
# print(os.listdir())
# uploaded = files.upload()

# **PDFs Convertor**

In [ ]:
import subprocess
import zipfile
from IPython.display import display, Image
import io

In [ ]:
# Function to download the converted files

def make_pdf(uploaded=None):
    zip_filename = "pdf_only.zip"
    converted = []

    def on_button_clicked(b):
        if os.path.exists(zip_filename):
            print("Download started...")
            files.download(zip_filename)
        else:
            print(f"File '{zip_filename}' not found.")

    download_btn = widgets.Button(
        description='Download ZIP',
        button_style='success',
        tooltip='Click to download the transformed slides in a ZIP file',
        icon='download'
    )

    def pptx_to_pdf(file_path):

        base_name = os.path.splitext(file_path)[0]
        ext = os.path.splitext(file_path)[1].lower()

        print(f"Converting {ext} to PDF via LibreOffice...")
        try :
            subprocess.run(
                ['soffice', '--headless', '--convert-to', 'pdf', file_path],
                check=True
            )
            lo_output = os.path.splitext(os.path.basename(file_path))[0] + ".pdf"
            final_pdf = base_name + ".pdf"

            if os.path.exists(lo_output) and lo_output != tmp_pdf:
                os.rename(lo_output, tmp_pdf)
                return final_pdf

            else:
                print("  ✗ Temporary PDF not created by LibreOffice.")
                return None
        except Exception as e:
            print(f"System Error: Could not convert {ext}. {e}")
            return None

        return final_pdf

    def convert_to_pdf(file_path, zip_filename):
        ext = os.path.splitext(file_path)[1].lower()

        if ext == '.json':
              converted.append(file_path)
              print(f"Download {file_path}: by clicking the button.\n")
              return None
        if ext in ('.ppt', '.pptx'):
            print(f"Converting PowerPoint: {file_path}...")
            result = pptx_to_pdf(file_path)
            if result:
                converted.append(result)
            else:
                print(f"  ✗ Failed to convert {file_path}\n")

        elif ext == '.pdf':
            print(f"Skipping {file_path}: Already in PDF format.\n")
            converted.append(file_path)
        else:
            print(f"Converting {file_path} to PDF...")
            subprocess.run(
                ['soffice', '--headless', '--convert-to', 'pdf', file_path],
                check=True
            )
            pdf_path = os.path.splitext(file_path)[0] + ".pdf"
            if os.path.exists(pdf_path):
                print(f"Successfully converted {file_path} to {pdf_path}")
                converted.append(pdf_path)
            else:
                print(f"Failed to convert {file_path}: Output PDF not found.\n")
                return None

    if uploaded is None:
        print("Upload your files here:\n")
        uploaded = files.upload()

    if isinstance(uploaded, str):
        uploaded = [uploaded]
    i = 1
    for filename in uploaded:
        print(f"\nFile {i} : {filename}")
        convert_to_pdf(filename, zip_filename)
        i += 1

    if converted:
        with zipfile.ZipFile(zip_filename, 'w') as zipf:
            for pdf in converted:
                zipf.write(pdf)
        download_btn.on_click(on_button_clicked)
        display(download_btn)
    else:
        print("The file type is not supported.")

## **Selective Text**

In [ ]:
#Function which extract texts from specific pages

def ext_text(filepath=None, page_list=None):
    extracted_texts = []
    if filepath is None:
        filepath = input("Enter the path of the PDF file: \n")
    if page_list is None:
        temp = input("Enter the pages to be extracted (comma-separated): \n")
        page_list = [int(page) for page in temp.split(',')]

    try:
      with open(filepath, 'r', encoding='utf-8') as f:
                  data = json.load(f)

      def get_flattened_nodes(nodes_list):
          flat = []
          for node in nodes_list:
              flat.append(node)
              if 'children' in node:
                  flat.extend(get_flattened_nodes(node['children']))
          return flat

      all_nodes = get_flattened_nodes(data.get('nodes', []))

      for page_num in page_list:
          page_content = [node.get('text', '') for node in all_nodes if node.get('page_index') == page_num]

          if page_content:
              # Concatenate all text segments for this page
              combined_text = "\n".join(filter(None, page_content)).strip()
              extracted_texts.append(combined_text)
              print(f"Extracted text from page {page_num}")
          else:
              extracted_texts.append("")
              print(f"Page {page_num} not found in the index. Skipping.")
    except Exception as e:
        print(f"Error processing JSON: {e}")
    return extracted_texts

# **API CALL PageIndex**

##**Use PageIndex API to compare the tree**

In [ ]:
# %pip install -q --upgrade pageindex requests openai PyMuPDF
# ! pip install -U pageindex
# import os
# from google.colab import files
# from google.colab import userdata
# from pageindex import PageIndexClient
# import pageindex.utils as utils

In [ ]:
# from google.colab import files
# doc_list = []
# pi_client = PageIndexClient(api_key="")        #Enter API here
# print(pi_client)
# print("Upload your files here:\n")
# uploaded = files.upload()

# for files in uploaded :
#   doc = "/content/"
#   doc += files

#   try:
#       result = pi_client.submit_document(doc)
#       print("API is valid:", result)
#       doc_id = result["doc_id"]
#       doc_list.append(doc_id)
#       if pi_client.is_retrieval_ready(doc_id):
#         tree = pi_client.get_tree(doc_id)['result']  #node_summary=True
#         print(f'Index tree has been created for {files} with doc_id = {doc_id}')
#         utils.print_tree(tree)
#       else:
#         print("Processing document, please wait")
#   except Exception as e:
#      print("API error:", e)

### **Tree to JSON Convertor**

In [ ]:
# #to print json tree
# import pprint
# pi_client = PageIndexClient(api_key="") #Enter API here
# import json
# doc_list = [""]    #Add the document ID like : pi-jjvjvj

# tree_json = []

# for doc_id in doc_list:
#   tree_result = pi_client.get_tree(doc_id, node_summary = False)
#   if tree_result.get("status") == "completed":

#     json_filename = f"{doc_id}.json"
#     tree_data = tree_result.get("result")

#     with open(json_filename, 'w', encoding='utf-8') as f:
#       json.dump(tree_data, f, indent=4)

#     tree_json.append(json_filename)
#     print(f"Saved tree structure for {doc_id} to {json_filename}\nAccess it in the list tree_json[]\n")

#     print("PageIndex Tree Structure: \n")
#     pprint.pprint(json.dumps(tree_result.get("result"), indent=4))

#**Tree Builder**

**Page Aware**

In [ ]:
#Function to map out the contents with its page

from pypdf import PdfReader
def build_slide_page_map(original_path: str, processing_path: str) -> tuple[dict[int, int], int]:
    pptx_path = original_path
    if original_path.lower().endswith('.ppt'):
        subprocess.run(
            ['soffice', '--headless', '--convert-to', 'pptx', original_path],
            check=True
        )
        pptx_path = os.path.splitext(original_path)[0] + '.pptx'

    from pptx import Presentation
    prs = Presentation(pptx_path)
    ppt_slide_count = len(prs.slides)

    pdf_pages = len(PdfReader(processing_path).pages)
    offset = pdf_pages - ppt_slide_count

    slide_map = {
        pdf_page: pdf_page - offset
        for pdf_page in range(1, pdf_pages + 1)
    }

    return slide_map, ppt_slide_count

In [ ]:
#Function to markdown content with its page

def export_markdown_with_page_markers(document) -> str:
    from docling_core.types.doc import TextItem, SectionHeaderItem, PictureItem, TableItem, DocItemLabel

    pages_text = {}

    for element, _ in document.iterate_items():
      if not hasattr(element, "prov") or not element.prov:
          continue

      # page_no must come FIRST
      page_no = getattr(element.prov[0], "page_no", 0)
      if page_no not in pages_text:
          pages_text[page_no] = []

      if isinstance(element, PictureItem):
          continue

      if isinstance(element, TableItem):
          try:
              table_md = element.export_to_markdown(doc=document)
              if table_md:
                  pages_text[page_no].append(table_md)
          except Exception:
              pass
          continue

      if not (hasattr(element, "text") and element.text):
          continue

        # Emit headings with # prefix so heading_re can match them
      is_heading = isinstance(element, SectionHeaderItem)
      if not is_heading and isinstance(element, TextItem):
          label = getattr(element, "label", None)
          if label in (DocItemLabel.SECTION_HEADER, "section_header", "heading"):
              is_heading = True

      if is_heading:
          pages_text[page_no].append(f"# {element.text.strip()}")
      else:
          pages_text[page_no].append(element.text)

    lines = []
    for page_no in sorted(pages_text.keys()):
        lines.append(f"\n<!-- PAGE:{page_no} -->\n")
        lines.extend(pages_text[page_no])
    return "\n".join(lines)

**Title Generator**

In [ ]:
#Function to generate a Tree based on Titles

import re

BULLET_CHARS = {"●", "•", "◦", "○", "▪", "▸", "→"}

def get_prefix(title: str) -> str:
    s = title.strip()
    if not s:
        return "plain"
    first = s[0]
    if first in BULLET_CHARS:
        return "bullet"
    if first.isdigit():
        return "digit"
    if first.islower():
        return "lower_alpha"
    return "plain"


def build_heading_tree(flat_headings: list[dict]) -> list[dict]:
    roots: list[dict] = []
    stack: list[tuple[dict, int]] = []
    dynamic_levels: dict[str, int] = {}
    next_level = 1

    for entry in flat_headings:
        node  = dict(entry)

        ptype = get_prefix(entry["title"])

        if ptype == "plain":
            level = 0
            dynamic_levels = {}
            next_level = 1
        else:
            if ptype not in dynamic_levels:
                dynamic_levels[ptype] = next_level
                next_level += 1
            elif stack and dynamic_levels[ptype] <= stack[-1][1]:
              if not any(get_prefix(s[0]["title"]) == ptype for s in stack):
                  dynamic_levels[ptype] = next_level
                  next_level += 1
            level = dynamic_levels[ptype]

        while stack and stack[-1][1] >= level:
            stack.pop()

        if stack:
            stack[-1][0].setdefault("children", []).append(node)
        else:
            roots.append(node)

        stack.append((node, level))
    return roots


def build_heading_table(document, slide_page_map: dict[int, int] = None, ppt_slide_count: int = None,) -> tuple[list[dict], int, list[dict]]:
    raw: list[dict] = []
    figures: list[dict] = []
    seen_pages: set[int] = set()
    doc_order = 0

    def to_slide(page_no: int) -> int:
      if page_no <= 0:
              return 0
      if slide_page_map:
          return max(1, slide_page_map.get(page_no, page_no))
      return page_no

    for element, _ in document.iterate_items():
        doc_order += 1
        if hasattr(element, "prov") and element.prov:
            for prov in element.prov:
                    seen_pages.add(prov.page_no)
        if isinstance(element, PictureItem):
            figure_entry = None
            page_no = 0
            if hasattr(element, "prov") and element.prov:
               page_no = getattr(element.prov[0], "page_no", 0)
            vlm_texts = [
                ann.text
                for ann in getattr(element, "annotations", [])
                if hasattr(ann, "text") and ann.text
            ]
            if vlm_texts:
              slide_no = to_slide(page_no)
              figure_entry = {
                  "title" : f"Figure (p.{slide_no})",
                  "page_index" : slide_no,
                  "text"  : "\n\n".join(vlm_texts),
                  "is_vlm": True
              }
            if figure_entry is not None:
                figure_entry["doc_order"] = doc_order
                figures.append(figure_entry)
            continue

        is_heading = False
        if isinstance(element, SectionHeaderItem):
            is_heading = True
        elif isinstance(element, TextItem):
            label = getattr(element, "label", None)
            if label in (DocItemLabel.SECTION_HEADER, "section_header", "heading"):
                is_heading = True
            elif get_prefix((element.text or "").strip()) in ("bullet", "digit", "lower_alpha"):
              if len((element.text or "").strip()) <= 60:
                  is_heading = True
        if not is_heading:
            continue

        title = (element.text or "").strip()
        if not title:
            continue

        page_no = 0
        if hasattr(element, "prov") and element.prov:
             page_no = getattr(element.prov[0], "page_no", 0)

        node_id = str(getattr(element, "self_ref", "") or "0000")
        raw.append({"title": title, "start": to_slide(page_no), "node_id": node_id, "doc_order": doc_order})

    total_pages = ppt_slide_count or (max(seen_pages) if seen_pages else 0)
    deduped: list[dict] = []
    for entry in raw:
        if deduped and deduped[-1]["title"] == entry["title"]:
            deduped[-1]["end_page"] = entry["start"]
            continue
        if deduped:
            deduped[-1]["end_page"] = entry["start"] - 1
        deduped.append(entry)

    if deduped:
      deduped[-1]["end_page"] = total_pages

    headings    = build_heading_tree(deduped)
    return headings, total_pages, figures

# def print_heading_table(headings: list[dict], total_pages: int) -> None:
#     print("\n" + "═" * 62)
#     print(f"  HEADING TABLE  (total pages in document: {total_pages})")
#     print("═" * 62)

#     def _print_node(node: dict, indent: int = 0) -> None:
#         pad    = "    " * indent
#         marker = "| " if indent > 0 else ""
#         print(f"  {pad}{marker}[p.{node['start']:>3}]  {node['title']}")
#         for child in node.get("children", []):
#             _print_node(child, indent + 1)

#     for root in headings:
#         _print_node(root)

#     print("═" * 62 + "\n")

**Tree Generator**

In [ ]:
#Function to generate tree based on Docling Extraction

import re
from itertools import count

def extract_section_texts(markdown: str) -> dict[tuple[str, int], str]:
    page_marker_re = re.compile(r'^<!-- PAGE:(\d+) -->$', re.MULTILINE)
    heading_re = re.compile(r'^#{1,6}\s+(.+)$', re.MULTILINE)
    page_splits = list(page_marker_re.finditer(markdown))
    sections: dict[tuple[str, int], str] = {}

    last_title = None

    for i, page_match in enumerate(page_splits):
        page_no    = int(page_match.group(1))
        page_start = page_match.end()
        page_end   = page_splits[i + 1].start() if i + 1 < len(page_splits) else len(markdown)
        page_text  = markdown[page_start:page_end]

        headings = list(heading_re.finditer(page_text))

        first_heading_start = headings[0].start() if headings else len(page_text)
        leading_content = page_text[:first_heading_start].strip()

        if leading_content and last_title:
            key = (last_title, page_no)
            existing = sections.get(key, "")
            sections[key] = (existing + "\n\n" + leading_content) if existing else leading_content

        for j, match in enumerate(headings):
            title      = match.group(1).strip()
            body_start = match.end()
            body_end   = headings[j + 1].start() if j + 1 < len(headings) else len(page_text)
            content    = page_text[body_start:body_end].strip()
            last_title = title
            key        = (title, page_no)

            if key in sections:
                sections[key] = sections[key] + "\n\n" + content
            else:
                sections[key] = content
    return sections


def build_ideal_output(
    tree_headings : list[dict],
    section_texts : dict[tuple[str, int], str],
    total_pages   : int,
    figures       : list[dict] = None,
) -> dict:

    target_headings = tree_headings
    node_counter = count(1)
    flat_nodes: list[dict] = []
    top_level_flat: list[dict] = []

    def enrich(node: dict, is_root: bool = False) -> dict:
        title   = node["title"]
        page_no = node.get("start", 0)
        end_page = node.get("end_page", page_no)

        content = (
            node.get("vlm_annotation")
            or node.get("text")
            or "\n".join(filter(None, [
                section_texts.get((title, p), "")
                for p in range(page_no, end_page + 1)
            ]))
            or next(
                    (v for (t, p), v in sorted(
                        section_texts.items(),
                        key=lambda x: abs(x[0][1] - page_no)
                    ) if t == title),
                    ""
            )
        )

        all_pages = node.get("all_pages", [page_no])
        enriched = {
            "title"      : title,
            "node_id"    : f"{next(node_counter):04d}",
            "page_index" : page_no,
            "doc_order"  : node.get("doc_order", 0),
            "text"       : f"{title}\n{content}" if content else title,
        }
        children = [enrich(c) for c in node.get("children", [])]
        if children:
            enriched["children"] = children
        if is_root:
            top_level_flat.append(enriched)
        flat_nodes.append(enriched)
        return enriched

    nodes = [enrich(n, is_root=True) for n in target_headings]
    flat_nodes.sort(key=lambda n: n.get("doc_order", 0))
    top_level_flat.sort(key=lambda n: n.get("doc_order", 0))

    for fig in (figures or []):
        fig_node = {
            "title"      : fig.get("title", "Figure"),
            "node_id"    : f"{next(node_counter):04d}",
            "page_index" : fig.get("page_index", 0),
            "doc_order"  : fig.get("doc_order", 0),
            "text"       : fig.get("vlm_annotation") or fig.get("text", ""),
        }
        parent = next(
            (n for n in reversed(flat_nodes) if n.get("doc_order", 0) < fig_node["doc_order"]),
            None,
        )
        if parent is not None:
            parent.setdefault("children", []).append(fig_node)
        else:
            nodes.insert(0, fig_node)

    def _strip_doc_order(node):
        node.pop("doc_order", None)
        for child in node.get("children", []):
            _strip_doc_order(child)

    for n in nodes:
        _strip_doc_order(n)

    return {"total_pages": total_pages, "nodes": nodes}

# **Layout Extraction**

In [ ]:
#Main Function to generate the final docling_report and tree

import io
import base64
import httpx
from collections.abc import Iterable
from google.colab import userdata
from IPython.display import Markdown, display
import time
import ipywidgets as widgets
import json

from docling_core.types.doc import DoclingDocument, NodeItem
from docling_core.types.doc.document import PictureItem, PictureDescriptionData
from docling_core.types.doc import SectionHeaderItem, TextItem
from docling_core.types.doc.labels import DocItemLabel

from docling.datamodel.base_models import InputFormat, ItemAndImageEnrichmentElement
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.models.base_model import BaseItemAndImageEnrichmentModel
from docling.pipeline.standard_pdf_pipeline import StandardPdfPipeline
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    EasyOcrOptions,
    PdfPipelineOptions,
    PictureDescriptionApiOptions,
    TableFormerMode,
    ChartExtractionModelOptions,
    LayoutOptions
)

# 1. API Configuration
from groq import Groq
VLM_API_KEY = userdata.get('VLM_API_KEY')
VLM_URL = "https://api.groq.com/openai/v1/chat/completions"
VISION_MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"
vlm_client = Groq(api_key=VLM_API_KEY)

sys_prompt = """
You are a precise document digitization engine. Your sole function is to extract
every visible piece of information from an image and represent it as structured Markdown.
Follow the specific rules after determining the type of the image.

#STEP 1 — IDENTIFY IMAGE TYPE
Silently classify the image as one of: UI Screen, Chart/Graph, Data Table, Technical Diagram, Formula, Code/Terminal, Photograph/Scan, or Legend/Key. Use this to guide output structure without stating it.

#STEP 2 — EXTRACTION RULES
## Text
- Extract ALL visible text verbatim
- Preserve exact spelling, capitalization, punctuation, and abbreviations exactly as shown
- If text is partially obscured, cut off, or ambiguous, reproduce what is visible and append [UNCLEAR]
- Never correct typos, spelling mistakes, or formatting errors — reproduce them as-is
## Structure & Hierarchy
- Reflect the visual hierarchy using Markdown heading levels (# ## ###)
- Reproduce nested structures using nested Markdown lists
- Preserve reading order as it appears spatially (top to bottom, left to right)

#STEP 3 — TYPE-SPECIFIC RULES
##UI Screens & Web Apps: Page/window title first → navigation/menus/tabs/breadcrumbs as lists → form fields with labels, values, placeholders, states (☑ ☐ selected disabled active) → buttons with exact labels → status messages, badges, timestamps → all table rows and columns.
##Chart/Graph/Plot: State chart type explicitly (bar, line, pie, scatter, histogram, etc.) → title → axis labels, titles, tick values, units → all data series names and values → For each data series, enumerate every visible data point as (x-label, y-value); estimate from grid lines if not labelled exactly; never omit or skip a data point → legend entries → annotations, callouts, trend/reference lines with labels and values.
##Data Table/Spreadsheet/Form: Reproduce as Markdown table → exact headers → every row/cell verbatim → For visually merged or spanned cells, fill each affected cell with [MERGED] → empty cells: [EMPTY] → For calendars and scheduling grids, treat each day/slot as a table cell
##Technical Diagram/Flowchart/Schematic: All labeled components with their exact → directional connections as A → B, bidirectional as A ↔ B → connector/arrow labels → Capture the label on every arrow or connector if one exists → Extract hierarchy as nested lists or numbered outline → Extract all callouts, tooltips, legends ##verbatim → For flowcharts, preserve decision branches and label each path (Yes/No, True/False, etc.)
##Formula/Math: Extract Verbatim in LaTeX notation → preserve subscripts, superscripts, Greek letters, operators as they are → Label each formula if a label or equation number is visible.
##Code/Terminal: Verbatim in fenced code blocks with language tag (```python, ```sql) → preserve indentation, spacing, line breaks → include line numbers, error messages, annotations.
##Photograph/Document Scan: One-sentence subject/setting description → all visible text verbatim (signs, labels, overlays, watermarks, stamps, handwriting) → note spatial layout of key elements (top-left, center, bottom-right, etc.) → For document scans, extract body text preserving paragraph breaks and section headings
##Legends, Keys & Footnotes: Extract legends and keys as a dedicated section at the end → Preserve every symbol-to-meaning mapping → Extract all footnotes, disclaimers, source attributions verbatim.

# Summary (Required — always last)
Generate a very detailed report using extracted information encompassing the semantic meaning of the image covering all key information:
- Name the subject, value, and context in every sentence
- Use varied verbs: "accounts for", "represents", "contributes", "totals", "shows"
- Phrase as direct answers to "what", "how much", "who", "which" — optimised for search retrieval
- Cross-reference multiple sections or charts where relevant
"""

class VLMEnricherPipelineOptions(PdfPipelineOptions):
    do_vlm_enrichment: bool = True

class DynamicVLMEnricher(BaseItemAndImageEnrichmentModel):

    images_scale = 2.5

    def __init__(self, enabled: bool, api_url: str, api_key: str, model: str, timeout: int = 60):
        self.enabled  = enabled
        self.api_url  = api_url
        self.api_key  = api_key
        self.model    = model
        self.timeout  = timeout
        self.last_figure_failed = False
        self._session = httpx.Client(timeout=self.timeout)

    def is_processable(self, doc: DoclingDocument, element: NodeItem) -> bool:
        return self.enabled and isinstance(element, PictureItem)

    def _reset_session(self):
      try:
          self._session.close()
      except Exception:
          pass
      self._session = httpx.Client(timeout=self.timeout)
      print("[VLM Enricher] 🔄 API session reset")

    def __call__(self, doc: DoclingDocument, element_batch: Iterable[ItemAndImageEnrichmentElement]) -> Iterable[NodeItem]:
        if not self.enabled:
            return

        for enrich_element in element_batch:
            element: PictureItem = enrich_element.item
            image                = enrich_element.image   # PIL Image

            print(f"\n[VLM Enricher] Figure detected → pausing pipeline...")

            # Classification label
            label = ""
            if hasattr(element, "classifications") and element.classifications:
                label = element.classifications[0].predicted_class or ""

        # Encode image
            try:
                buf = io.BytesIO()
                image.save(buf, format="PNG")
                b64 = base64.b64encode(buf.getvalue()).decode()
            except Exception as e:
                print(f"[VLM Enricher] ⚠ Could not encode image: {e}")
                yield element
                continue

            # Call VLM
            figure_had_failure = False
            while True:
                try:
                    completion = vlm_client.chat.completions.create(
                            model=VISION_MODEL,
                            messages=[
                                {
                                    "role": "user",
                                    "content": [
                                        {
                                            "type": "text",
                                            "text": sys_prompt
                                        },
                                        {
                                            "type": "image_url",
                                            "image_url": {
                                                "url": f"data:image/png;base64,{b64}"
                                            }
                                        }
                                    ]
                                }
                            ],
                            temperature=0.3,
                            max_completion_tokens=1024,
                            top_p=1,
                            stream=False,
                            stop=None,
                        )
                    annotation = completion.choices[0].message.content
                    #time.sleep(4)
                    print(f"[VLM Enricher] ✓ Annotation received ({len(annotation)} chars)")
                    element.annotations.append(
                        PictureDescriptionData(text=annotation, provenance="vlm-enricher")
                    )
                    break   # ← success, exit loop

                except Exception as e:
                    wait_time = 600
                    print(f"[VLM Enricher] ⚠ VLM call failed: {e}")

                    resp = getattr(e, "response", None)
                    status_code = getattr(resp, "status_code", None) or getattr(e, "status_code", None)

                    print(f"[VLM Enricher] ⚠ VLM call failed (HTTP {status_code}): {e}")
                    if status_code == 429 and resp is not None:
                        print(dict(resp.headers))

                    bar = widgets.IntProgress(value=wait_time, min=0, max=wait_time,
                                              description='Cooldown:',
                                              bar_style='warning',
                                              style={'bar_color': '#0047AB'},
                                              layout=widgets.Layout(width='400px'))
                    lbl = widgets.Label(value=f'{wait_time}s remaining')
                    box = widgets.HBox([bar, lbl])
                    display(box)
                    print(f"You got a 10 min break, get some water!! ;-)\n")
                    for remaining in range(wait_time, 0, -1):
                        bar.value = remaining
                        lbl.value = f'{remaining}s remaining'
                        time.sleep(1)
                    bar.value     = 0
                    bar.bar_style = 'success'
                    lbl.value     = 'Retrying...'
                    time.sleep(0.4)
                    box.close()

                    self.last_figure_failed = figure_had_failure
                    self._reset_session()
                    print(f"[VLM Enricher] 🔄 Retrying...\n")

            print(f"[VLM Enricher] ✓ Written back — resuming pipeline\n")
            yield element


class VLMEnricherPipeline(StandardPdfPipeline):

    def __init__(self, pipeline_options: VLMEnricherPipelineOptions):
        super().__init__(pipeline_options)
        self.pipeline_options: VLMEnricherPipelineOptions

        self.enrichment_pipe = [DynamicVLMEnricher(
                enabled  = self.pipeline_options.do_vlm_enrichment,
                api_url  = VLM_URL,
                api_key  = VLM_API_KEY,
                model    = VISION_MODEL)]

        if self.pipeline_options.do_vlm_enrichment:
            self.keep_backend = True

    @classmethod
    def get_default_options(cls) -> VLMEnricherPipelineOptions:
        return VLMEnricherPipelineOptions()


pipeline_options = VLMEnricherPipelineOptions(do_ocr=True,
ocr_options=EasyOcrOptions(
    confidence_threshold=0.3,
    force_full_page_ocr=True,
),
generate_picture_images=True,
images_scale=3.0,
do_picture_description=False,
do_picture_classification=True,
do_chart_extraction=True,
chart_extraction_options=ChartExtractionModelOptions(),
do_table_structure=True,
enable_remote_services=True,
do_code_enrichment=True,
do_formula_enrichment=True,
do_vlm_enrichment=True)

pipeline_options.layout_options = LayoutOptions(
    confidence_threshold=0.3
)

pipeline_options.table_structure_options.mode = TableFormerMode.ACCURATE

def layout_ext(file_path = None):
    if file_path is None:
      file_path = input("Enter the file path (PDF or PPTX):\n").strip()
      if not os.path.exists(file_path):
          print("Error: File not found.")
          return

    base_name, ext = os.path.splitext(file_path)
    ext = ext.lower()
    processing_path = file_path

    if ext in ['.ppt', '.pptx']:
        print(f"Converting {ext} to PDF via LibreOffice...")
        try:
            subprocess.run(['soffice', '--headless', '--convert-to', 'pdf', file_path], check=True)
            processing_path = f"{base_name}.pdf"
        except Exception as e:
            print(f"System Error: Could not convert {ext}. {e}")
            return
    if ext in ['.ppt', '.pptx']:
        slide_map, ppt_slide_count = build_slide_page_map(file_path, processing_path)
    else:
        slide_map, ppt_slide_count = None, None

    print("=" * 60)
    print(f"Extracting with Docling + VLM : ({VISION_MODEL})")
    print("=" * 60)

    converter = DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_cls=VLMEnricherPipeline,pipeline_options=pipeline_options,)})

    print("Processing document")
    result = converter.convert(processing_path)

    # Export to Markdown
    markdown_output = result.document.export_to_markdown()
    page_marked_markdown = export_markdown_with_page_markers(result.document)
    my_filename = f"{base_name}_report.md"
    with open(my_filename, "w", encoding="utf-8") as f:
        f.write(markdown_output)
        f.close()
    doc_op.insert(0,my_filename)

    print("── Building Hierarchial tree ──────────────────────────────────────────────────────────────\n")
    headings, total_pages, figures = build_heading_table(result.document,slide_page_map=slide_map, ppt_slide_count=ppt_slide_count,)
    section_texts = extract_section_texts(page_marked_markdown)
    ideal_output  = build_ideal_output(headings, section_texts, total_pages, figures)
    tree = f"{base_name}_tree.json"
    with open(tree, "w", encoding="utf-8") as f:
        json.dump(ideal_output, f, indent=2, ensure_ascii=False)
    print(f"Ideal JSON saved    → {tree}")

    tree_json.insert(0, tree)
    print("── Can download Docling markdown file and tree ──────────────────────────────────────────────────────────────\n")


In [ ]:
# import os
# print(os.listdir())
# uploaded = files.upload()

# **Embedding**

In [ ]:
#Function to ping Jina.ai

# import requests
# import json
# import pprint
# from google.colab import userdata

# EMBEDDING_API_KEY = userdata.get('EMBEDDING_API_KEY')

# url = "https://api.jina.ai/v1/embeddings"
# headers = {
#     "Content-Type": "application/json",
#     "Authorization": EMBEDDING_API_KEY
# }
# data = {
#     "model": "jina-embeddings-v5-text-small",
#     "task": "text-matching",
#     "truncate": True,
#     "normalized": True,
#     "input": [
#         "Ferrari 812 superfast"]
# }

# response = requests.post(url, headers=headers, data=json.dumps(data))
# pprint.pprint(response.json())

In [ ]:
#Function which embeds the text and store it in Chroma DB

import json
import os
import pprint
import requests
import time
from google.colab import userdata

# ── Global counter
doc_counter = 0

EMBEDDING_API_KEY = userdata.get('EMBEDDING_API_KEY')
embedding_model = "jina-embeddings-v5-text-small"
MODEL_URL     = "https://api.jina.ai/v1/embeddings"
CHUNK_SIZE   = 512          #Can handle upto 8192 tokens
CHUNK_OVERLAP = 16


# ── 1. Chunker
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap   # slide forward with overlap
    return chunks if chunks else [text]


# ── 2. Embedding call
def get_embeddings(texts: list[str]) -> list[list[float]]:
    payload = {
        "model": embedding_model,
        "task": "text-matching",
        "truncate": True,
        "normalized": True,
        "input": texts
    }
    response = requests.post(
        MODEL_URL,
        headers={"Content-Type": "application/json", "Authorization": EMBEDDING_API_KEY},
        json=payload
    )
    response.raise_for_status()
    data = response.json()


    items = sorted(data["data"], key=lambda x: x["index"])
    #time.sleep(0.6)
    return [item["embedding"] for item in items]


# ── 3. Main extraction + DB insertion
def embed(filename, collection):
    global doc_counter
    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)
    tp = data.get("total_pages", 0)


    # ── JSON traversal
    def extract_all_text(node, results=None):
        if results is None:
            results = []
        texts = {field: node[field] for field in ["text"] if field in node}
        if texts:
            results.append({
                "node_id":    node.get("node_id", "N/A"),
                "page_index": node.get("page_index", "N/A"),
                **texts
            })
        for child in node.get("children", []):
            extract_all_text(child, results)
        return results

    def process_json(file_path):
        if not os.path.exists(file_path):
            print(f"Error: File '{file_path}' not found.")
            return []
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        all_texts = []
        if isinstance(data, dict):
            root_nodes = data.get("nodes", [])
        elif isinstance(data, list):
            root_nodes = data
        else:
            print("Error: JSON root must be an object or an array of objects.")
            return []

        for root_node in root_nodes:
            all_texts.extend(extract_all_text(root_node))

        # Assign next_index
        for i in range(len(all_texts) - 1):
            all_texts[i]["next_index"] = all_texts[i + 1]["page_index"]
        if all_texts:
            all_texts[-1]["next_index"] = None
        return all_texts


    results = process_json(filename)
    print(f"Found {len(results)} node(s) in '{filename}'.\n")
    for node in results:
        chunks      = chunk_text(node["text"])
        total       = len(chunks)
        curr_index  = node["page_index"]
        next_index  = node["next_index"] if node["next_index"] is not None else tp

        print(f"  node_id: {node['node_id']} → {total} chunk(s) | "
              f"curr_index: {curr_index} | next_index: {next_index}")

        # ── API Call
        embeddings = get_embeddings(chunks)

        # ── Storage
        for chunk_i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
            doc_counter += 1
            doc_id = f"doc{doc_counter}"

            collection.add(
                ids        = [doc_id],
                documents  = [chunk],
                embeddings = [embedding],      # ← pre-computed embedding
                metadatas  = [{
                    "node_id":    node["node_id"],
                    "curr_index": curr_index,
                    "next_index": next_index,
                    "filename":   os.path.basename(filename)
                }]
            )
            print(f"{doc_id} | chunk {chunk_i + 1}/{total}")

    print(f"\nDone. Total docs in DB: {doc_counter}")

# **Vector DB**

In [ ]:
import chromadb

In [ ]:
#Intializing Database

def innit_vector_db () :
  clientdb = chromadb.Client()
  #clientdb.delete_collection("demo")
  collection = clientdb.get_or_create_collection(name="demo")
  return collection

In [ ]:
#Embedding the Content

def vector_db():
  collection = innit_vector_db()
  for tree in tree_json:
    embed(tree, collection)

# **Retrival**

In [ ]:
import pprint
from collections import defaultdict

In [ ]:
#In house Logic to retrieve top - k nodes

def get_nodes(result):

    ret_nodes = []
    dist = result.get("distances", [[]])[0]
    metadatas = result.get("metadatas", [[]])[0]
    ret_pages = []

    top_5 = [(m["curr_index"], m["next_index"]) for m in metadatas[:5]]

    filtered_data = [
        (d, m) for d, m in zip(dist, metadatas)
        if d <= 0.85
    ]

    if filtered_data:
        freq_map = defaultdict(lambda: {"all_freq": 0})
        for _, m in filtered_data:
            key = (m["curr_index"], m["next_index"])
            freq_map[key]["all_freq"] += 1

        valid_ranges = {
            k for k, v in freq_map.items()
            if v["all_freq"] > 1 or k in top_5
        }

        seen = set()
        ret_nodes = [
            m["node_id"]
            for _, m in filtered_data
            if (m["curr_index"], m["next_index"]) in valid_ranges
            and not (m["node_id"] in seen or seen.add(m["node_id"]))
        ]

        ret_pages = list(dict.fromkeys(
            m["curr_index"]
            for _, m in filtered_data
            if m["node_id"] in seen
        ))

        ret_display = list(dict.fromkeys(
          page
          for _, m in filtered_data
          if m["node_id"] in seen
          for page in range(m["curr_index"], m["next_index"] + 1)
          ))

    return ret_nodes,ret_pages,ret_display #Size down ret_nodes and ret_pages if you are hitting token limitations upon inference

In [ ]:
#To check retrieved pages withouth any limitations

def display_distance():
  query = input("Enter your query: \n")
  collection = innit_vector_db()

  results1 = collection.query(
      query_embeddings = get_embeddings([query]),  # ← use for queries too
      n_results        = 10,
      include          = ["distances", "metadatas"]
  )
  pprint.pprint(results1)


In [ ]:
#display_distance()

# **Inference**

In [ ]:
import os
from google.colab import files
from google.colab import userdata
from IPython.display import Markdown, display

In [ ]:
#Function to fetch relevant text and call LLM for inference

def llm_call (question):

  LLM_API_KEY = userdata.get('LLM_API_KEY')
  model="openai/gpt-oss-120b",
  client = Groq(api_key=LLM_API_KEY)

  sys_prompt = """
  Rules = { Strictly refer to given text only. At the beginning of the response state/display all the relevant data from the source as it is with the page number of the document for each acquired data, and then under
  the "Inference" section, explain every possible scenario that can be predicted by using the procured data. Do not make up any information of your own, and do not search any other resources for the answer.
  Do reply with "I dont know the answer saaaar :-(" if the relevant information is not present within the given information. }
  """
  print(f"\nProcessing the querry: \n{question}")

  collection = innit_vector_db()
  results = collection.query(
      query_embeddings = get_embeddings([question]),  # ← use for queries too
      n_results        = 10,
      include          = ["distances", "metadatas"]
  )

  selected = get_nodes(results)

  print(f"Selected Pages for the Query are : {selected[2]}\nNodes selected are : {selected[0]}\n\n")
  # make_pdf()
  temp = ext_text(tree_json[0], selected[1])  #Demo2
  addoc = '\n'.join(temp)
  question = "Answer the following Question : " + question

  query = sys_prompt + addoc + question
  completion = client.chat.completions.create(
      model="openai/gpt-oss-120b",
      messages=[
          {
              "role": "user",
              "content": [
                  {
                      "type": "text",
                      "text": query
                  },
              ],
          }
      ]
  )
  display(Markdown(completion.choices[0].message.content))

In [ ]:
#Function for the user

def ask_question():
  from collections import deque
  import json

  LLM_API_KEY = userdata.get('LLM_API_KEY')
  model="openai/gpt-oss-120b",
  client = Groq(api_key=LLM_API_KEY)

  queries = []

  system_prompt = """
  Rules = { Split the user query into individual questions if it contains multiple questions. Strictly do not change the text.
  Resolve all pronoun references (e.g. "their", "its", "they") by replacing them with the explicit subject from the query context.
  Return the input if no changes are need. Return a JSON array of strings. No question marks. No extra text.}
  """
  user_prompt = input("Enter your query: \n")

  query = system_prompt + user_prompt
  try :
      completion = client.chat.completions.create(
          model="openai/gpt-oss-120b",
          messages=[
              {
                  "role": "user",
                  "content": [
                      {
                          "type": "text",
                          "text": query
                      },
                  ],
              }
          ]
      )
  except (e):
    print(e)

  parsed_queries = json.loads(completion.choices[0].message.content)
  queries = deque(parsed_queries)
  for query in queries:
      llm_call(query)
      print("\n")


#**Main.exe**

###**Works best for single file**
###**Will bring Multi-file indexing capabilities soon**

In [ ]:
print("Upload the pdf/ppt/docx file to be indexed by the RAG pipeline\n")
uploaded = files.upload()

print("\n\n── Detecting Layout of the file ──────────────────────────────────────────────────────────────")
if isinstance(uploaded, str):
        uploaded = [uploaded]
i = 1
for filename in uploaded:
    print(f"\nFile {i} : {filename}")
    layout_ext(filename)
    i += 1

make_pdf(uploaded = [doc_op[0], tree_json[0]])
#Download all the created documents at once
# make_pdf(uploaded = doc_op + tree_json)

print("── Using Jina AI to embedd and Chroma DB to store ──────────────────────────────────────────────────────────────\n")
innit_vector_db()
vector_db()

print("── Document has been indexed, Ask questions now ──────────────────────────────────────────────────────────────\n")

In [ ]:
print("── Ask Questions ──────────────────────────────────────────────────────────────\n")
choice = 1
while choice == 1:
  ask_question()

  choice = int(input("Enter 1 to ask question again!! ;-)\t"))